In [ ]:
from time import perf_counter  

from modeling import build_models_from_csv
from bundles import BaseBundle, MSBundle
from utils import (
    tighten_bounds_one_model,
    MIN_DIST, ACTIVE_TOL, GAP_STOP_TOL,
)
from simplex import run_pid_simplex_3d

RUN_QUICK_TEST = True

if RUN_QUICK_TEST:
    csv_path       = "data.csv"
    max_scenarios  = 1
    target_nodes   = 13
else:
    csv_path       = "data.csv"
    max_scenarios  = 99
    target_nodes   = 30

bounds = {
    "x":  (None, None),
    "u":  (None, None),
    "e":  (None, None),
    "I":  (-10, 10),
    "Kp": (-10, 10),
    "Ki": (0, 100),
    "Kd": (0, 1000),
}

'''
bounds={
    "Kp": (-10.0, 10.0),
    "Ki": (-100.0, 100.0),
    "Kd": (-100.0, 1000.0),
    "x": (-2.5, 2.5),
    "u": (-5.0, 5.0),
    "e": (-1e6, 1e6),
    "I": (-1e6, 1e6),
}
'''
weights = (1.0, 0.01)

# ====== stage 1：load data and generate models ======
t_load0 = perf_counter()
model_list, first_stg_vars_list, m_tmpl_list, T = build_models_from_csv(
    csv_path, h=0.2, weights=weights, bounds=bounds,
    sp0=0.0, sp1=0.5, ku_col="tau_us", tau_col="tau_xs",
    disturb_prefix="disturbance_", setpoint_change_col="setpoint_change",
    max_scenarios=max_scenarios, skip=0
)
t_load1 = perf_counter()
print(f"[Time] Data load & scenario build: {t_load1 - t_load0:.3f}s")

# ====== stage 1.5：FBBT / OBBT ======
# same as snog: FBBT open,OBBT open
obbt_solver_opts = {
    "NonConvex": 2,
    "MIPGap": 1,     
    "TimeLimit": 5   
}
for m, yvars in zip(model_list, first_stg_vars_list):
    tighten_bounds_one_model(m, yvars,
                             use_fbbt=True,
                             use_obbt=True,
                             obbt_solver_name="gurobi",
                             obbt_solver_opts=obbt_solver_opts,
                             max_rounds=3, tol=1e-6, verbose=True)

# ====== stage 2: Persistent Solver Packaging ======
ub_options = {
    'NonConvex': 2,        
}
lb_options = {
    'NonConvex': 2,
    'MIPGap': 0.2,            
    'TimeLimit': 15           
}

t_wrap0 = perf_counter()
base_bundles = [BaseBundle(m, ub_options) for m in model_list]  
ms_bundles   = [MSBundle(m, yvars, lb_options) for m, yvars in zip(model_list, first_stg_vars_list)]  # LB 侧
t_wrap1 = perf_counter()
print(f"[Time] Persistent wrapper (GurobiPersistent) setup: {t_wrap1 - t_wrap0:.3f}s")

# ====== stage 3：main loop ======
agg_bundle = None  

t_run0 = perf_counter()
hist = run_pid_simplex_3d(
    base_bundles=base_bundles,
    ms_bundles=ms_bundles,
    model_list=model_list,
    first_vars_list=first_stg_vars_list,
    target_nodes=target_nodes,
    min_dist=MIN_DIST,
    active_tol=ACTIVE_TOL,
    verbose=True,
    agg_bundle=agg_bundle,
    gap_stop_tol=1e-1,   
)
t_run1 = perf_counter()
print(f"[Time] Main loop total: {t_run1 - t_run0:.3f}s")

# ====== print output ======
print("\n==== Done ====")
print(f"Total nodes: {len(hist['nodes'])}")
print(f"Best UB: {min(hist['UB_hist']) if hist['UB_hist'] else None}")
print(f"Last LB: {hist['LB_hist'][-1] if hist['LB_hist'] else None}")

[Time] Data load & scenario build: 0.003s
[Tighten] rounds=1, changed=False
Set parameter MIPGap to value 0.1
Set parameter NumericFocus to value 1
Set parameter Presolve to value 2
Set parameter NonConvex to value 2
Set parameter MIPGap to value 0.2
Set parameter NumericFocus to value 1
Set parameter Presolve to value 2
Set parameter NonConvex to value 2
Set parameter TimeLimit to value 15
[Time] Persistent wrapper (GurobiPersistent) setup: 0.018s
containing a solution
containing a solution
[Iter 0] created=6 (cum=6), active=4, active+UB=2, ms_recomputed=6
[Iter 0] Optimality gap: 7.764328e-02 (50.220%)
[Iter 0] Active simplex ratio = 0.666667
[Iter 0] UB node (10.0, 100.0, 0.0) is in simplices [3, 4]
[Iter 0] LB = 0.076962 = UB(0.154606) + ms_b(-7.764e-02) from T4
[Iter 0] candidate rank #1: T4, scene=0, ms=-7.764e-02
== ms candidates (sorted by (ms, -dist)) ==
rank   simp   scene           ms    mind(all)                             pt
-----------------------------------------------

[Iter 0] Elapsed: 36.121s
containing a solution
containing a solution
containing a solution
containing a solution
[Iter 1] created=12 (cum=18), active=8, active+UB=8, ms_recomputed=12
[Iter 1] Optimality gap: 1.051293e-01 (88.681%)
[Iter 1] Active simplex ratio = 0.684308
[Iter 1] UB node (-1.9213512627652316, 89.39978052923337, 0.15604057785588515) is in simplices [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
[Iter 1] LB = 0.013419 = UB(0.118548) + ms_b(-1.051e-01) from T8
[Iter 1] candidate rank #1: T8, scene=0, ms=-1.051e-01
== ms candidates (sorted by (ms, -dist)) ==
rank   simp   scene           ms    mind(all)                             pt
----------------------------------------------------------------------------
   1     T8       0  -1.0513e-01     4.93e+01  (-0.3411, 48.2947, 1000.0000)
   2     T9       0  -1.0513e-01     4.93e+01  (-0.3407, 48.2967, 1000.0000)
   3     T3       0  -7.7000e-02     1.86e+00     (-1.7485, 87.5514, 0.0000)
   4     T4       0  -7.5639e-02     6.64e+0

WARNING (W1002): Setting Var 'Kd' to a numeric value `1000.0000000000005`
outside the bounds (0, 1000).
    See also https://pyomo.readthedocs.io/en/stable/errors.html#w1002
[Iter 1] Elapsed: 63.721s
containing a solution
containing a solution
containing a solution
containing a solution
